In [7]:
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents

from ingest import build_index
from rag_helper import RAGPgVector
from utilities import calculate_llm_price

from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
openai_client = OpenAI(
    # base_url="http://127.0.0.1:1234/v1",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    # api_key="1234",
)

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

q1 = "How does approximate nearest neighbor search work?"
v1 = model.encode(q1)
v1[0]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

np.float32(-0.020582033)

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [3]:
len(documents)

72

In [12]:
for d in documents:
    if "02-vector-search/lessons/07-sqlitesearch-vector.md" == d["filename"]:
        req_content = d["content"]

req_content

'# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is ex

In [13]:
v2 = model.encode(req_content)

In [14]:
v2.dot(v1)

np.float32(0.45681328)

In [15]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [16]:
len(chunks)

295

In [ ]:
from tqdm.auto import tqdm
X = []
for chunk in tqdm(chunks):
    

  0%|          | 0/295 [00:00<?, ?it/s]

# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.

Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your phone uses a simple language model for that. It predicts 